# 07 – Phân cụm: K-Means, Agglomerative, DBSCAN/HDBSCAN, t-SNE

**Người thực hiện:** thinh (Nguyễn Vĩnh Thịnh)  
**Mục tiêu:** Trả lời RQ3: phân cụm không giám sát tái tạo cấu trúc ngành hàng tới đâu?

**Quy tắc:** chạy TỪNG ô một, đổi tham số rồi chạy lại; mỗi ngày làm xong thì `git add` + `git commit -m "<mô tả đúng việc đã làm>"` + `git push`. Cuối buổi xuất HTML: `python ../scripts/export_notebook_html.py <notebook> thinh`.

**Các bước:**
1. K-Means: quét K (6–20) và số chiều SVD.
2. Agglomerative (mẫu 5.000–8.000 điểm): linkage ward/average/complete.
3. DBSCAN: quét eps, min_samples; HDBSCAN: min_cluster_size.
4. t-SNE trên mẫu 5.000 điểm, tô màu theo ngành và theo cụm.
5. Chọn thuật toán tốt nhất, giải thích Silhouette/ARI/NMI bằng lời.

In [3]:
import sys, json, pandas as pd, numpy as np, matplotlib.pyplot as plt
sys.path.append('../src')
from run_experiment import run
from figlog import save_fig
from exp_logger import log_generic
from common import load_data, clean_title


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, HDBSCAN
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score

df = load_data(); df['clean'] = df.title.map(clean_title); y = df.category_label.values
Xt = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True).fit_transform(df.clean)

def embed(m):
    return Normalizer().fit_transform(TruncatedSVD(m, random_state=42).fit_transform(Xt))

def score(Z, lab, algo, **cfg):
    rng = np.random.RandomState(42); s = rng.choice(len(Z), min(10000, len(Z)), replace=False)
    ok = lab[s] != -1  # bỏ điểm nhiễu của DBSCAN khi tính silhouette
    r = dict(algo=algo, **cfg, n_clusters=len(set(lab)) - (1 if -1 in lab else 0), noise_pct=float((lab == -1).mean()),
             silhouette=silhouette_score(Z[s][ok], lab[s][ok], metric='cosine') if len(set(lab[s][ok])) > 1 else float('nan'),
             ari=adjusted_rand_score(yy, lab), nmi=normalized_mutual_info_score(yy, lab))
    log_generic('thinh', 'cluster_log', r); print(r); return r

## 1. K-Means

In [7]:
Z = embed(50); yy = y
lab = KMeans(12, n_init=10, random_state=42).fit_predict(Z)
score(Z, lab, 'kmeans', K=12, svd=50)

{'algo': 'kmeans', 'K': 12, 'svd': 50, 'n_clusters': 12, 'noise_pct': 0.0, 'silhouette': 0.3051387794838741, 'ari': 0.5135807898260646, 'nmi': 0.6848373686875174}


{'algo': 'kmeans',
 'K': 12,
 'svd': 50,
 'n_clusters': 12,
 'noise_pct': 0.0,
 'silhouette': 0.3051387794838741,
 'ari': 0.5135807898260646,
 'nmi': 0.6848373686875174}

> **Nhật ký lần chạy** – ngày giờ đã tự ghi trong `experiments/thinh/`. Sau khi xem kết quả, mở file CSV và **tự viết** cột `nhan_xet` (thấy gì?) và `rut_ra` (quyết định gì cho lần sau?). Viết lại ngắn gọn ở ô này:
>
> - Kết quả chính:0.2936
> - Nhận xét:0.5636
> - Lần sau thử:k=8,K=12

## 2. Agglomerative (trên mẫu)

In [ ]:
idx = np.random.RandomState(0).choice(len(df), 6000, replace=False)
Zs, yy = embed(50)[idx], y[idx]
lab = AgglomerativeClustering(10, linkage='ward').fit_predict(Zs)
score(Zs, lab, 'agglomerative', K=10, linkage='ward', n_sample=6000)

> **Nhật ký lần chạy** – ngày giờ đã tự ghi trong `experiments/thinh/`. Sau khi xem kết quả, mở file CSV và **tự viết** cột `nhan_xet` (thấy gì?) và `rut_ra` (quyết định gì cho lần sau?). Viết lại ngắn gọn ở ô này:
>
> - Kết quả chính: …
> - Nhận xét: …
> - Lần sau thử: …

## 3. DBSCAN / HDBSCAN

In [ ]:
lab = DBSCAN(eps=0.3, min_samples=10, metric='euclidean').fit_predict(Zs)
score(Zs, lab, 'dbscan', eps=0.3, min_samples=10, n_sample=6000)

> **Nhật ký lần chạy** – ngày giờ đã tự ghi trong `experiments/thinh/`. Sau khi xem kết quả, mở file CSV và **tự viết** cột `nhan_xet` (thấy gì?) và `rut_ra` (quyết định gì cho lần sau?). Viết lại ngắn gọn ở ô này:
>
> - Kết quả chính: …
> - Nhận xét: …
> - Lần sau thử: …

In [ ]:
lab = HDBSCAN(min_cluster_size=30).fit_predict(Zs)
score(Zs, lab, 'hdbscan', min_cluster_size=30, n_sample=6000)

> **Nhật ký lần chạy** – ngày giờ đã tự ghi trong `experiments/thinh/`. Sau khi xem kết quả, mở file CSV và **tự viết** cột `nhan_xet` (thấy gì?) và `rut_ra` (quyết định gì cho lần sau?). Viết lại ngắn gọn ở ô này:
>
> - Kết quả chính: …
> - Nhận xét: …
> - Lần sau thử: …

## 4. t-SNE

In [ ]:
from sklearn.manifold import TSNE
P = TSNE(2, perplexity=30, random_state=42, init='pca').fit_transform(Zs)
fig, ax = plt.subplots(figsize=(7, 6))
for c in np.unique(yy):
    ax.scatter(*P[yy == c].T, s=3, label=c)
ax.legend(markerscale=4, fontsize=7); ax.set_title('t-SNE (mẫu 6.000) tô màu theo ngành')
save_fig(fig, 'thinh_tsne_category', notebook='07_Thinh_PhanCum.ipynb', member='thinh', cell='Mục 4', y_nghia='(tự viết)')

## 5. Kết luận của Thịnh (tự viết)
- Thuật toán tốt nhất: … (run_id …)
- Silhouette = … nghĩa là …
- NMI = … nghĩa là …